# 🍷 Esteira de Machine Learning — Dataset Wine

| Campo | Valor |
|---|---|
| **Aluno** | Paulo Gabriel Alves Dos Santos |
| **Professor** | Leonardo Villani |
| **Prazo** | 27 de maio de 2026 |
| **Modelo** | XGBoost Classifier |
| **Dataset** | Wine Recognition (sklearn / UCI) |

---

**Objetivo:** Classificar tipos de vinho com base em 13 propriedades químicas usando XGBoost.

**Classes:** `class_0` | `class_1` | `class_2` — **178 amostras, 13 features**

---

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn -q
print('✅ Bibliotecas instaladas!')

---
## 📚 1. Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
import warnings; warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
np.random.seed(42)
print('✅ Importações realizadas!')

---
## 📂 2. Carregamento dos Dados

In [ ]:
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target
df['classe'] = df['target'].map({0: 'class_0', 1: 'class_1', 2: 'class_2'})
print(f'✅ Dataset Wine carregado: {df.shape[0]} amostras x {df.shape[1]} colunas')
display(df.head(10))

---
## 🔍 3. Exploração dos Dados (EDA)

In [ ]:
print('=' * 55)
print('  📊 INFORMAÇÕES DO DATASET WINE')
print('=' * 55)
print(f'\n🔹 Amostras  : {len(df)}')
print(f'🔹 Features  : {len(wine.feature_names)}')
print(f'🔹 Classes   : {list(wine.target_names)}')
print('\n📋 Tipos de dados:')
print(df.dtypes)
print('\n📋 Estatísticas descritivas:')
display(df.describe().round(3))
print('\n⚠️  Valores nulos por coluna:')
print(df.isnull().sum())
print('\n📊 Distribuição das classes:')
print(df['classe'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distribuição das classes
contagem = df['classe'].value_counts().sort_index()
cores = ['#2196F3', '#4CAF50', '#FF9800']
contagem.plot(kind='bar', ax=axes[0], color=cores, edgecolor='black', alpha=0.85)
axes[0].set_title('Distribuição das Classes de Vinho', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Classe'); axes[0].set_ylabel('Quantidade')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(contagem): axes[0].text(i, v+0.5, str(v), ha='center', fontweight='bold')

# Mapa de correlação entre features
corr = df[list(wine.feature_names)].corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0, ax=axes[1],
            linewidths=0.3, square=True)
axes[1].set_title('Mapa de Correlacao entre Features', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_wine.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico EDA salvo: eda_wine.png')

---
## 🔄 4. TRANSFORMAÇÃO EM COLUNAS 

**Transformação realizada:** Normalização da feature 'alcohol' (escala 0-1)

**Por quê?** Normalizar garante que features em escalas diferentes não dominem o modelo.

In [ ]:
print('=' * 60)
print('ITEM 4: TRANSFORMAÇÃO EM COLUNAS')
print('=' * 60)

# Mostrar ANTES
print('\n📊 ANTES da transformação:')
print(f'   Coluna: alcohol')
print(f'   Min: {df["alcohol"].min():.3f}')
print(f'   Max: {df["alcohol"].max():.3f}')
print(f'   Média: {df["alcohol"].mean():.3f}')
print(f'   Primeiras 5 valores: {list(df["alcohol"].head().round(2))}')

# Aplicar transformação: NORMALIZAR 'alcohol'
scaler = MinMaxScaler()
df['alcohol'] = scaler.fit_transform(df[['alcohol']])

# Mostrar DEPOIS
print('\n✅ DEPOIS da transformação (normalização MinMax):')
print(f'   Coluna: alcohol')
print(f'   Min: {df["alcohol"].min():.3f}')
print(f'   Max: {df["alcohol"].max():.3f}')
print(f'   Média: {df["alcohol"].mean():.3f}')
print(f'   Primeiras 5 valores: {list(df["alcohol"].head().round(2))}')

print('\n✅ Transformação concluída! A feature "alcohol" foi normalizada para escala 0-1.')

---
## ✂️ 5. TRANSFORMAÇÃO EM LINHAS

**Transformação realizada:** Remoção de outliers na feature 'total_phenols' usando método IQR

**Por quê?** Outliers podem prejudicar o treinamento do modelo.

In [ ]:
print('=' * 60)
print('ITEM 5: TRANSFORMAÇÃO EM LINHAS')
print('=' * 60)

n_antes = len(df)
print(f'\n📊 ANTES da transformação:')
print(f'   Total de linhas: {n_antes}')
print(f'   Coluna analisada: total_phenols')

# Calcular IQR (Interquartile Range) para detectar outliers
Q1 = df['total_phenols'].quantile(0.25)
Q3 = df['total_phenols'].quantile(0.75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

print(f'\n   Q1 (25%): {Q1:.3f}')
print(f'   Q3 (75%): {Q3:.3f}')
print(f'   IQR: {IQR:.3f}')
print(f'   Limite inferior: {limite_inf:.3f}')
print(f'   Limite superior: {limite_sup:.3f}')

# Remover outliers
df = df[(df['total_phenols'] >= limite_inf) & (df['total_phenols'] <= limite_sup)]

n_depois = len(df)
n_removidas = n_antes - n_depois

print(f'\n✅ DEPOIS da transformação (remoção de outliers):')
print(f'   Total de linhas: {n_depois}')
print(f'   Linhas removidas: {n_removidas}')
print(f'   Percentual removido: {n_removidas/n_antes*100:.1f}%')

print('\n✅ Transformação concluída! Outliers foram removidos.')

---
## 🧹 6. Limpeza Adicional e Preparação dos Dados

In [ ]:
print('🧹 Limpeza adicional dos dados...')
tam_inicial = df.shape[0]

n_dup = df.duplicated().sum()
print(f'  • Duplicatas   : {n_dup}', '→ removidas' if n_dup > 0 else '(nenhuma) ✅')
if n_dup > 0: df = df.drop_duplicates()

n_nulos = df.isnull().sum().sum()
print(f'  • Valores nulos: {n_nulos}', '→ removidos' if n_nulos > 0 else '(nenhum) ✅')
if n_nulos > 0: df = df.dropna()

print(f'\nTamanho: {tam_inicial} → {df.shape[0]} amostras')

X = df[list(wine.feature_names)].copy()
y = df['target'].copy()
print(f'\n📊 X (features): {X.shape}')
print(f'📊 y (alvo)    : {y.shape}')
print(f'📊 Classes     : {sorted(y.unique())} → {list(wine.target_names)}')

---
## ✂️ 7. Divisão em Treino / Validação / Teste

| Conjunto | Proporção |
|---|---|
| **Treino** | 60% |
| **Validação** | 20% |
| **Teste** | 20% |

In [ ]:
# Passo 1: 80% treino+val | 20% teste
X_temp, X_teste, y_temp, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

# Passo 2: do 80% → 75% treino | 25% val (= 60% e 20% do total)
X_treino, X_val, y_treino, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

total = len(X_treino) + len(X_val) + len(X_teste)
print('✅ Divisão concluída!')
print(f'  • Treino    : {len(X_treino):3d} amostras ({len(X_treino)/total*100:.0f}%)')
print(f'  • Validação : {len(X_val):3d} amostras ({len(X_val)/total*100:.0f}%)')
print(f'  • Teste     : {len(X_teste):3d} amostras ({len(X_teste)/total*100:.0f}%)')
print(f'  • Total     : {total} amostras')

---
## 🤖 8. Treinamento do Modelo XGBoost

In [ ]:
print('🤖 Treinando XGBoost...')
modelo = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)
modelo.fit(X_treino, y_treino,
           eval_set=[(X_treino, y_treino), (X_val, y_val)],
           verbose=False)
print('✅ Modelo treinado!')
print(f'  • n_estimators  : {modelo.n_estimators}')
print(f'  • max_depth     : {modelo.max_depth}')
print(f'  • learning_rate : {modelo.learning_rate}')

---
## 📊 9. Avaliação do Modelo

In [ ]:
y_pred_val = modelo.predict(X_val)
acuracia_val = accuracy_score(y_val, y_pred_val)
print('📊 RESULTADO — VALIDAÇÃO')
print('=' * 45)
print(f'🎯 Acurácia: {acuracia_val*100:.2f}%')
print('\n📋 Relatório por classe:')
print(classification_report(y_val, y_pred_val, target_names=wine.target_names))

In [ ]:
y_pred_teste = modelo.predict(X_teste)
acuracia_teste = accuracy_score(y_teste, y_pred_teste)
print('📊 RESULTADO FINAL — TESTE')
print('=' * 45)
print(f'🏆 Acurácia Final: {acuracia_teste*100:.2f}%')
print('\n📋 Relatório Completo:')
print(classification_report(y_teste, y_pred_teste, target_names=wine.target_names))

In [ ]:
cm = confusion_matrix(y_teste, y_pred_teste)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=wine.target_names, yticklabels=wine.target_names,
            linewidths=0.8, annot_kws={'size': 16}, ax=axes[0])
titulo0 = f'Matriz de Confusao (n) | Acuracia: {acuracia_teste*100:.2f}%'
axes[0].set_title(titulo0, fontsize=13, fontweight='bold')
axes[0].set_xlabel('Previsto', fontsize=12); axes[0].set_ylabel('Real', fontsize=12)

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Greens',
            xticklabels=wine.target_names, yticklabels=wine.target_names,
            linewidths=0.8, annot_kws={'size': 14}, ax=axes[1])
axes[1].set_title('Matriz de Confusao (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Previsto', fontsize=12); axes[1].set_ylabel('Real', fontsize=12)

plt.suptitle('XGBoost — Dataset Wine', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('matriz_confusao_wine.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Matriz salva: matriz_confusao_wine.png')

---
## 📈 10. Importância das Features

In [ ]:
importancias = modelo.feature_importances_
features = list(wine.feature_names)
indices = np.argsort(importancias)[::-1]

fig, ax = plt.subplots(figsize=(12, 5))
palette = plt.cm.Blues(np.linspace(0.4, 0.9, len(features)))
bars = ax.bar(range(len(features)), importancias[indices],
              color=palette[::-1], edgecolor='black', alpha=0.9)
for bar, imp in zip(bars, importancias[indices]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{imp:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(range(len(features)))
ax.set_xticklabels([features[i] for i in indices], rotation=30, ha='right', fontsize=9)
ax.set_title('Importancia das Features — XGBoost (Wine)', fontsize=14, fontweight='bold')
ax.set_xlabel('Feature'); ax.set_ylabel('Importancia')
plt.tight_layout()
plt.savefig('feature_importance_wine.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: feature_importance_wine.png')

---
## 🔮 11. PREDIÇÃO DO MODELO IMPLANTADO

Fazer predições com dados novos (que o modelo nunca viu)

In [ ]:
print('=' * 60)
print('ITEM 9: PREDIÇÃO DO MODELO IMPLANTADO')
print('=' * 60)

# Pegar alguns exemplos do conjunto de teste
print('\n📊 Exemplo 1 - Predições individuais:')
for i in range(5):
    exemplo = X_teste.iloc[i:i+1]
    pred = modelo.predict(exemplo)[0]
    pred_proba = modelo.predict_proba(exemplo)[0]
    real = y_teste.iloc[i]
    classe_pred = wine.target_names[pred]
    classe_real = wine.target_names[real]
    acerto = '✅' if pred == real else '❌'
    
    print(f'\n  Amostra {i+1}:')
    print(f'    Predito: {classe_pred}')
    print(f'    Real:    {classe_real}')
    print(f'    Confiança: {pred_proba.max()*100:.1f}%')
    print(f'    Resultado: {acerto}')

print('\n✅ Predições realizadas com sucesso!')

---
## 🏁 12. Conclusão

In [ ]:
print('=' * 70)
print('  RESUMO — ESTEIRA DE MACHINE LEARNING COMPLETA')
print('=' * 70)
print(f'\n  Aluno    : Paulo Gabriel Alves Dos Santos')
print(f'  Dataset  : Wine Recognition (sklearn / UCI)')
print(f'  Modelo   : XGBClassifier')
print(f'\n  REQUISITOS ATENDIDOS:')
print(f'\n  [✅] 1. Escolher base de dados (Wine)')
print(f'  [✅] 2. Registrar na planilha')
print(f'  [✅] 3. Estatísticas descritivas (Seção 3 - EDA)')
print(f'  [✅] 4. Transformação em COLUNAS (Seção 4 - Normalização alcohol)')
print(f'  [✅] 5. Transformação em LINHAS (Seção 5 - Remoção outliers)')
print(f'  [✅] 6. Dividir em treino/validação/teste (Seção 7)')
print(f'  [✅] 7. Treinar modelo (Seção 8)')
print(f'  [✅] 8. Matriz de confusão e acurácia (Seção 9)')
print(f'  [✅] 9. Predição do modelo implantado (Seção 11)')
print(f'\n  RESULTADOS:')
print(f'     Acuracia Validacao : {acuracia_val*100:.2f}%')
print(f'     Acuracia Teste     : {acuracia_teste*100:.2f}%')
print(f'\n  DATASET:')
print(f'     Amostras originais : 178')
print(f'     Amostras após transformações : {len(df)}')
print(f'     Features : {X.shape[1]}')
print(f'     Classes  : class_0, class_1, class_2 (tipos de vinho)')
print(f'\n' + '=' * 70)
print('  ✅ PROJETO CONCLUÍDO COM SUCESSO!')
print('=' * 70)